# 04 - Delphos search parameters

This notebook explains the knobs in `agent.propose(...)` and when to use them.


## Main parameters

| Parameter | Meaning | Practical use |
| --- | --- | --- |
| `n_models` | Number of unique model specifications to return | Increase for broader search |
| `max_attempts` | Maximum search attempts before stopping | Increase when duplicates are common |
| `strategy` | `greedy`, `stochastic`, `boltzmann`, or `topk` | Controls exploration |
| `epsilon` | Random-action probability for stochastic search | Higher means more random exploration |
| `temperature` | Softmax temperature for boltzmann/top-k sampling | Higher means more diversity |
| `top_k` | Candidate action pool for top-k search | Higher means broader local search |
| `horizon_kappa` | Search depth multiplier over attributes | Higher allows longer model edits |
| `linear_additive` | Start from a linear additive model | Set false to start from null model |
| `estimate` | Call Apollo/R environment | Use after proposal settings look good |
| `seed` | Reproducible stochastic search | Use for reports and comparisons |


## 1. Load baseline objects


In [1]:
import delphos as dp

agent = dp.load_agent(device="cpu")
task = dp.load_dataset("Swissmetro")


## 2. Greedy search

Greedy is deterministic and useful as a baseline. It usually returns one dominant path.


In [2]:
greedy = agent.propose(
    task,
    n_models=1,
    strategy="greedy",
)
greedy.to_dataframe()


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices
0,4,Swissmetro,1110_2212_3210_4214_5000_6110_7000,10,greedy,0,False,None,5,"[215, 17, 201, 125, 21, 215, 17, 27, 201, 73]"


## 3. Top-k search

Top-k is a good default for final users: it stays near high-value actions while still producing diverse models.


In [3]:
topk = agent.propose(
    task,
    n_models=5,
    max_attempts=80,
    strategy="topk",
    top_k=5,
    temperature=0.8,
    seed=42,
)
topk.to_dataframe()


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices
0,4,Swissmetro,1110_2124_3212_4111_5000_6126_7000,10,topk,0,False,None,5,"[21, 27, 17, 125, 74, 215, 27, 21, 75, 106]"
1,4,Swissmetro,1110_2124_3211_4214_5000_6110_7000,10,topk,1,False,None,5,"[17, 125, 21, 73, 27, 17, 74, 149, 21, 125]"
2,4,Swissmetro,1110_2124_3210_4110_5000_6110_7000,10,topk,2,False,None,5,"[215, 27, 21, 106, 17, 74, 105, 73, 201, 21]"
3,4,Swissmetro,1110_2120_3212_4214_5000_6110_7000,10,topk,3,False,None,5,"[215, 74, 201, 73, 21, 125, 74, 75, 27, 17]"
4,4,Swissmetro,1110_2120_3210_4111_5000_6126_7000,10,topk,4,False,None,5,"[17, 73, 125, 149, 27, 106, 17, 21, 215, 17]"


## 4. Boltzmann search

Boltzmann samples from all valid actions after weighting by Q-value. Increase temperature for more exploration.


In [4]:
boltzmann = agent.propose(
    task,
    n_models=5,
    max_attempts=80,
    strategy="boltzmann",
    temperature=1.2,
    seed=43,
)
boltzmann.to_dataframe()


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices
0,4,Swissmetro,1110_2210_3312_4211_5000_6314_7000,10,boltzmann,0,False,None,5,"[26, 225, 50, 19, 91, 45, 237, 123, 122, 25]"
1,4,Swissmetro,1110_2120_3314_4110_5000_6211_7000,10,boltzmann,1,False,None,5,"[37, 21, 42, 239, 21, 17, 71, 218, 75, 93]"
2,4,Swissmetro,1110_2313_3213_4124_5000_6116_7000,10,boltzmann,2,False,None,5,"[137, 12, 81, 207, 11, 13, 91, 44, 117, 76]"
3,4,Swissmetro,1110_2220_3314_4113_5000_6112_7000,10,boltzmann,3,False,None,5,"[17, 205, 119, 108, 41, 209, 92, 93, 33, 203]"
4,4,Swissmetro,1114_2121_3310_4120_5000_6313_7000,10,boltzmann,4,False,None,5,"[58, 18, 91, 140, 92, 131, 113, 236, 5, 89]"


## 5. Stochastic search

Stochastic search is epsilon-greedy. It is useful for stress-testing the action space.


In [5]:
stochastic = agent.propose(
    task,
    n_models=5,
    max_attempts=80,
    strategy="stochastic",
    epsilon=0.15,
    seed=44,
)
stochastic.to_dataframe()


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices
0,4,Swissmetro,1110_2120_3212_4110_5000_6126_7000,10,stochastic,0,False,None,5,"[215, 82, 201, 17, 215, 27, 201, 75, 17, 215]"
1,4,Swissmetro,1110_2212_3322_4214_5000_6126_7000,10,stochastic,1,False,None,5,"[215, 17, 201, 125, 21, 215, 17, 27, 101, 99]"
2,4,Swissmetro,1110_2212_3110_4214_5000_6110_7000,10,stochastic,2,False,None,5,"[215, 17, 201, 125, 21, 42, 215, 17, 27, 201]"
3,4,Swissmetro,1110_2212_3210_4110_5000_6126_7000,10,stochastic,3,False,None,5,"[215, 17, 201, 210, 27, 201, 73, 17, 215, 27]"
4,4,Swissmetro,1110_2120_3114_4110_5000_6110_7000,10,stochastic,4,False,None,5,"[215, 17, 58, 201, 21, 215, 27, 61, 201, 17]"


## 6. Search depth

`horizon_kappa` multiplies the number of task attributes to set a maximum number of actions. Larger values allow more edits but may create more complex specifications.


In [6]:
shallow = agent.propose(task, n_models=3, strategy="topk", horizon_kappa=1.0, seed=45)
deep = agent.propose(task, n_models=3, strategy="topk", horizon_kappa=3.0, seed=45)

print("Shallow episode lengths:", shallow.to_dataframe()["episode_length"].tolist())
print("Deep episode lengths:", deep.to_dataframe()["episode_length"].tolist())


Shallow episode lengths: [5, 5, 5]
Deep episode lengths: [15, 15, 15]


## 7. Start from null instead of linear additive

The default workflow starts from a linear additive specification. Set `linear_additive=False` when you want Delphos to build up from an empty model.


In [7]:
from_null = agent.propose(
    task,
    n_models=3,
    strategy="topk",
    linear_additive=False,
    seed=46,
)
from_null.to_dataframe()


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices
0,4,Swissmetro,1000_2000_3000_4000_5000_6220_7000,10,topk,0,False,None,1,"[6, 224, 210, 211, 218, 252, 220, 209, 219, 232]"
1,4,Swissmetro,1000_2110_3000_4000_5000_6121_7000,10,topk,1,False,None,2,"[6, 210, 225, 218, 226, 219, 211, 252, 217, 2]"
2,4,Swissmetro,1000_2000_3000_4326_5000_6000_7000,10,topk,2,False,None,1,"[4, 131, 126, 136, 138, 122, 124, 121, 142, 158]"


## 8. Custom checkpoints

The default checkpoint is bundled. Advanced users can pass another checkpoint trained elsewhere.


In [8]:
# custom_agent = dp.load_agent("/path/to/latest_checkpoint.pt", device="cpu")
# custom_models = custom_agent.propose(task, n_models=10)
